# Contextual Bandits: Theory and Intuition

## 1. Why Contextual Bandits?

Imagine you run a re-engagement campaign.  A user has gone dormant, and you want to send them a push notification to bring them back.  You have $K$ message variants — same underlying offer, different tone.  The question is: **which message should you send to *this particular* user?**

Three natural approaches, in order of sophistication:

**A/B testing** picks a single best message for *all* users.  Run a randomized experiment, measure CTR per variant, deploy the winner.  This ignores context — maybe "Personalized picks" is best for power shoppers while "We miss you" is best for long-tenured casual browsers.  A/B testing finds the globally best arm but leaves personalization gains on the table.

**Supervised learning** could learn a personalized model: estimate $p(\text{click} \mid x, a)$ for each user-message pair and pick the $\arg\max$.  But we only have labels for the message each user *actually received*.  Train on this data naively and the model will reinforce its own biases — a feedback loop we unpack in Section 5.

**Contextual bandits** solve both problems.  They learn a personalized policy — a mapping from user features to a distribution over actions — while *actively exploring* to collect the data needed to improve.

### Formal setup

| Component | Symbol | Description |
|-----------|--------|-------------|
| **Context** | $x \in \mathbb{R}^d$ | Feature vector describing the user ($d$ features) |
| **Action** | $a \in \{1, \dots, K\}$ | Which message variant to send ($K$ variants) |
| **Reward** | $r \in \{0,1\}$ | Click (1) or ignore (0) |
| **Policy** | $\pi(a \mid x)$ | A probability distribution over actions, conditioned on context |

Each round proceeds as: observe context $x$, sample action $a \sim \pi(\cdot \mid x)$, observe reward $r$ **for the chosen action only**.

This last constraint — **bandit feedback** — is the defining feature.  We never see what *would* have happened under a different action.  This is the fundamental **counterfactual problem**, and it is what separates bandits from supervised learning (where every example comes with its label) and what necessitates the importance-weighting machinery developed in Sections 4 and 5.


## 2. The Simulation Environment

To build intuition we work with a synthetic environment where we *know* the ground truth.  In practice the ground truth is unknown — that's the whole point — but having it lets us verify that the bandit is learning correctly and measure regret.

### Scenario

A dormant user arrives.  We choose one of $K = 4$ message variants:

| Action | Message | Intuition |
|--------|---------|-----------|
| 1 | "See what's new since you left" | Curiosity — highlights fresh content |
| 2 | "Your personalized picks await" | Personalization — leverages purchase history |
| 3 | "We miss you! Come back" | Emotional — relationship-driven appeal |
| 4 | "Trending picks just for you" | Social proof — what others are buying |

### Ground-truth click model

Each (user, message) pair has a click probability governed by a weight matrix $W_{\text{TRUE}} \in \mathbb{R}^{K \times d}$, where row $a$ contains the coefficients for action $a$:

$$
p(\text{click} \mid x, a) \;=\; \sigma\!\bigl(W_{\text{TRUE}}[a] \cdot x\bigr)
$$

where $\sigma(z) = \frac{1}{1 + e^{-z}}$ is the sigmoid, $W_{\text{TRUE}}[a] \in \mathbb{R}^d$ is the weight vector for action $a$, and $x = [1, \;\text{lifetime\_purchases\_std}, \;\text{days\_as\_customer\_std}]$ (so $d = 3$).

These are **independent Bernoulli means** — they do *not* sum to 1 across actions.  Each is "how likely is a click *if* we send this particular message to this particular user."

### The bias term and baseline CTRs

The leading 1 in $x$ is a bias/intercept.  Each row of $W_{\text{TRUE}}$ has the form $[\text{bias}, \; w_{\text{purchases}}, \; w_{\text{days}}]$, so the logit expands to:

$$
z_a(x) = \underbrace{W[a,0]}_{\text{bias}} + W[a,1] \cdot \text{purchases\_std} + W[a,2] \cdot \text{days\_std}
$$

At the population mean (both standardized features = 0), the feature terms vanish and each action's click rate collapses to $\sigma(\text{bias})$:

| Action | Bias | Baseline CTR |
|--------|------|-------------|
| "See what's new" | $-2.5$ | 7.6% |
| "Personalized picks" | $-2.5$ | 7.6% |
| "We miss you" | $-2.8$ | 5.7% |
| "Trending picks" | $-2.2$ | 10.0% |

These are realistic for re-engagement notifications (~6–10%).  The feature weights then shift CTRs up or down depending on the user's profile — which the bandit must discover without knowing $W_{\text{TRUE}}$.

### Oracle policy

The **oracle** always picks the action maximizing true CTR:

$$
a^*(x) = \arg\max_a \; W_{\text{TRUE}}[a] \cdot x
$$

We can drop $\sigma$ because it is strictly monotonic — it preserves the ranking.  The boundary between any two actions $a$ and $b$ is where their logits are equal: $(W_{\text{TRUE}}[a] - W_{\text{TRUE}}[b]) \cdot x = 0$, a linear equation in $x$.  So the oracle partitions the feature space into convex regions — one per winning action — separated by straight lines.

> **The bandit's job** is to discover these regions through exploration and reward feedback, *without* ever knowing $W_{\text{TRUE}}$.  The gap between the oracle's CTR and the bandit's CTR is the **regret**.


## 3. Policies and Exploration

A **policy** $\pi$ maps each context $x$ to a probability distribution over $K$ actions.  The agent samples an action from this distribution, so there is always some controlled randomness.  This randomness is essential: it ensures we sometimes try suboptimal-looking actions, which is the only way to discover they might actually be better.

### The interaction loop

On every round:

1. **Observe context** — a dormant user's features arrive as $x$.
2. **Predict & explore** — the policy outputs $\pi(\cdot \mid x)$ and we sample $a \sim \pi$.  The probability $\pi(a \mid x)$ assigned to the sampled action is recorded — this is the **propensity**.
3. **Observe reward** — the environment reveals whether the user clicked ($r = 1$) or not ($r = 0$) for the chosen action only.
4. **Learn** — the tuple $(x, a, r, \pi(a \mid x))$ is used to update the model.

### $\varepsilon$-greedy exploration

The simplest exploration strategy.  Let $\hat{r}(x, a)$ be the model's current estimate of the reward for action $a$ in context $x$.  Then:

$$
\pi(a \mid x) =
\begin{cases}
1 - \varepsilon + \dfrac{\varepsilon}{K} & \text{if } a = \arg\max_{a'} \hat{r}(x, a') \\[6pt]
\dfrac{\varepsilon}{K} & \text{otherwise}
\end{cases}
$$

The logic: with probability $\varepsilon$, explore (pick uniformly at random); with probability $1 - \varepsilon$, exploit (pick the estimated best).  The formula folds both into a single distribution — the greedy action gets the exploitation mass *plus* its share of the exploration mass.

**Concrete example** ($K = 4$, $\varepsilon = 0.1$):

| Action | Probability |
|--------|------------|
| Best (greedy) | $0.9 + 0.025 = 0.925$ |
| Each other | $0.025$ |

**Boundary cases:**
- $\varepsilon = 1$: uniform random — every action gets $1/K$.  Pure exploration, no learning signal used.
- $\varepsilon = 0$: always pick the greedy best.  Pure exploitation — no new information collected.

In practice we start with $\varepsilon = 1$ (collect unbiased data), then reduce $\varepsilon$ to exploit what we've learned.

### The model inside the policy

The $\hat{r}(x, a)$ in the $\varepsilon$-greedy formula is a learned reward estimator.  In VowpalWabbit, this is a linear model with a **separate weight vector $\hat{W}[a]$ for each action**, predicting:

$$
\hat{r}(x, a) = \hat{W}[a] \cdot x
$$

This means we are learning $K$ independent linear models — one per action.  Each action gets its own bias and its own coefficients for every context feature.  For instance, the "Personalized picks" model can learn that purchase history is strongly predictive, while the "We miss you" model can independently learn that account age matters most.  The actions do not share parameters.

**Why not a single shared model?**  An alternative is to learn one weight vector $w$ over a joint feature vector $\phi(x, a)$ — say, $x$ concatenated with a one-hot encoding of the action.  Without interaction features this reduces to $\hat{r}(x, a) = w_{\text{shared}} \cdot x + b_a$: all actions share the same context coefficients and differ only by an intercept.  That model can learn that some actions have higher baseline CTR, but it **cannot** learn that a given feature helps one action while hurting another — exactly the signal the bandit needs.  To recover full expressiveness you would need to include all action $\times$ feature crosses, which gives $K \times d$ parameters — the same count as $K$ separate weight vectors, just rearranged.  Per-action weights are the simpler way to get there.

This is a raw linear score — no sigmoid.  The model doesn't need calibrated probabilities; it only needs to **rank** actions correctly for each context.  Since $\sigma$ is strictly monotonic, correct linear rankings $\Rightarrow$ correct probability rankings.


## 4. Off-Policy Evaluation: Estimating What Would Have Happened

Suppose we have logged data collected under one policy and want to estimate the performance of a *different* policy — without deploying it.  This is **off-policy evaluation (OPE)**, and it is central to how contextual bandits work in practice.

### The setup

Two policies are in play:

| Symbol | Name | Role |
|--------|------|------|
| $\mu(a \mid x)$ | **Logging (behavior) policy** | The policy that *actually chose* the actions when the data was collected |
| $\pi(a \mid x)$ | **Target (candidate) policy** | A different policy whose performance we want to estimate |

We have a log of $N$ interactions (where $N$ is the total number of observations collected), indexed $i = 1, \dots, N$:

$$
\{(x_i, a_i, r_i, \mu_i)\}_{i=1}^N
$$

where each tuple records the context $x_i$, the action $a_i$ that $\mu$ chose, the observed reward $r_i$, and the propensity $\mu_i = \mu(a_i \mid x_i)$ — the probability with which $\mu$ selected that action.

We want to estimate $V(\pi) = \mathbb{E}_{x,\, a \sim \pi}[r(x,a)]$, the expected reward we would get if we had used $\pi$ instead.

### The core idea: importance weighting

The logged data reflects $\mu$'s choices, not $\pi$'s.  If $\mu$ chose action $a_i$ for user $x_i$ and we observed reward $r_i$, the question is: **how much more (or less) often would $\pi$ have chosen that same action?**

The **importance weight** answers this:

$$
w_i = \frac{\pi(a_i \mid x_i)}{\mu(a_i \mid x_i)}
$$

- $w_i > 1$ means $\pi$ would have chosen $a_i$ *more often* than $\mu$ did — upweight this observation.
- $w_i < 1$ means $\pi$ would have chosen $a_i$ *less often* — downweight it.
- $w_i = 0$ means $\pi$ would *never* choose $a_i$ — discard the observation entirely.

**Grounding example.**  Suppose $\mu$ is uniform ($\varepsilon = 1$, so $\mu(a \mid x) = 0.25$ for all actions) and $\pi$ is a near-greedy policy that puts probability $0.925$ on action 2 and $0.025$ on the rest.  For a logged observation where $\mu$ happened to pick action 2: $w_i = 0.925 / 0.25 = 3.7$ — this observation counts almost 4× because $\pi$ strongly favors that action.  For one where $\mu$ picked action 3: $w_i = 0.025 / 0.25 = 0.1$ — nearly discarded, because $\pi$ would rarely go there.

This reweighting only works because we recorded $\mu(a_i \mid x_i)$ — the propensity — at data-collection time.  Without it, off-policy evaluation is impossible.  This is why recording propensities is non-negotiable.

### IPS (Inverse Propensity Scoring)

Using the importance weights $w_i$ defined above (one per logged observation $i = 1, \dots, N$), we estimate $\pi$'s expected reward by reweighting each observation:

$$
\hat{V}_{\text{IPS}}(\pi) = \frac{1}{N} \sum_{i=1}^{N} w_i \cdot r_i
$$

Each term $w_i \cdot r_i$ takes the observed reward $r_i$ and scales it by how much more (or less) likely $\pi$ was to have chosen that action compared to $\mu$.  Averaging over all $N$ observations gives the estimate.

**Sanity check — when $\pi = \mu$.**  If we evaluate the same policy that collected the data, every weight is $w_i = \mu(a_i \mid x_i) / \mu(a_i \mid x_i) = 1$, and IPS reduces to $\frac{1}{N}\sum_i r_i$ — the plain sample-mean CTR.  This is the right answer: the best estimate of a policy's reward from its own data is just the observed average.

**Why this gives the expected reward — a worked example.**  Consider $K = 2$ actions and $N = 1000$ observations logged under uniform $\mu$ (50/50):

| | Action 1 | Action 2 |
|---|---|---|
| Times chosen by $\mu$ | 500 | 500 |
| Clicks observed | 100 | 50 |
| Observed CTR | 20% | 10% |

The average CTR under $\mu$ is $150/1000 = 15\%$.  Now suppose $\pi$ puts 90% on action 1 and 10% on action 2.  If we had actually deployed $\pi$, we'd expect ~900 pulls of action 1 ($\to$ 180 clicks) and ~100 pulls of action 2 ($\to$ 10 clicks), giving a CTR of $190/1000 = 19\%$.

IPS gets there without deploying $\pi$.  The weights are $w = 0.9/0.5 = 1.8$ for action 1 and $w = 0.1/0.5 = 0.2$ for action 2:

$$\hat{V}_{\text{IPS}} = \frac{1}{1000}\bigl[1.8 \times 100 + 0.2 \times 50\bigr] = \frac{180 + 10}{1000} = 19\%$$

What happened?  The weights **rebalanced the dataset** to match what $\pi$'s data would have looked like.  We had 500 action-1 observations, but each counts as 1.8 — effectively 900.  We had 500 action-2 observations, but each counts as 0.2 — effectively 100.  The reweighted dataset has the same action composition that $\pi$ would have generated.  And since the reward for a given (user, action) pair is determined by the environment — not by which policy selected it — a dataset with $\pi$'s action frequencies gives $\pi$'s expected reward.

**IPS is unbiased**: $\mathbb{E}_\mu[\hat{V}_{\text{IPS}}(\pi)] = V(\pi)$.  Each term's expectation under $\mu$ equals the corresponding expectation under $\pi$, via the standard importance sampling identity.

**The variance problem**: when $\mu(a_i \mid x_i)$ is small (the logging policy rarely chose this action), $w_i$ can blow up, and a single observation can dominate the entire sum.  This makes the estimate unstable in practice.  Returning to our example: if the logging policy only chose action 2 with probability $0.05$ instead of $0.25$, that single observation's weight would be $0.925 / 0.05 = 18.5$ — one data point now has 18× the influence of a typical observation.

### SNIPS (Self-Normalized IPS)

Normalize by the sum of weights:

$$
\hat{V}_{\text{SNIPS}}(\pi) = \frac{\displaystyle\sum_{i=1}^{N} w_i \cdot r_i}{\displaystyle\sum_{i=1}^{N} w_i}
$$

This introduces a small bias (it is no longer exactly unbiased) but dramatically reduces variance.  The denominator $\sum w_i$ acts as a normalizing constant: if one weight $w_i$ is huge, it inflates both numerator and denominator, so its influence on the ratio is dampened.

**Intuition**: IPS computes a weighted sum and divides by $N$ (the number of observations).  SNIPS computes the same weighted sum but divides by the sum of the weights instead — effectively asking "what fraction of the *reweighted* mass came from clicks?"  When all weights are equal (i.e. $\pi = \mu$, so every $w_i = 1$), the denominator equals $N$ and both estimators give the same answer: the plain sample-mean CTR.

### Computing $\pi(a \mid x)$ in practice

To compute the importance weight $w_i$, we need the value $\pi(a_i \mid x_i)$ — the probability the candidate policy would have assigned to the action that was actually taken.  For an $\varepsilon$-greedy policy, this is fully determined by three things: the model weights $\hat{W}$, the value of $\varepsilon$, and the context $x_i$.  No sampling is involved — we just evaluate the formula.

Concretely: the model computes $\hat{r}(x_i, a')$ for all $K$ actions, identifies the greedy best, and the $\varepsilon$-greedy formula (Section 3) directly gives $\pi(a \mid x_i)$ for every action.  For instance, with $\varepsilon = 0.05$ and $K = 4$:

- If $a_i$ is the greedy best action for context $x_i$: $\pi(a_i \mid x_i) = 0.95 + 0.05/4 = 0.9625$
- If $a_i$ is any other action: $\pi(a_i \mid x_i) = 0.05/4 = 0.0125$

That number goes straight into $w_i = \pi(a_i \mid x_i) / \mu(a_i \mid x_i)$.  We never need to actually *run* the candidate policy or sample from it — we just evaluate the probability it assigns to the action that was logged.

### Why this matters for contextual bandits

OPE lets us compare candidate policies *offline* — using only logged data.  After collecting data under a uniform-random policy ($\mu(a \mid x) = 1/K$), we can use SNIPS to estimate the CTR of various $\varepsilon$-greedy policies and pick the best $\varepsilon$ before deploying anything.  This is exactly what Phase 2 of the pipeline (Section 6) does.


## 5. Online Learning: Doubly Robust Estimation

OPE (Section 4) evaluates policies after the fact using logged data.  Now we turn to how the model **learns** on each round — specifically, how it updates its weights given bandit feedback where we only observe the outcome for one action.

### Why not just train a classifier?

A natural approach: treat this as supervised learning.  We have features $(x, a)$ and labels $r \in \{0,1\}$.  Train $\hat{r}(x, a) \approx p(\text{click} \mid x, a)$, then at decision time pick $\arg\max_a \hat{r}(x, a)$.

This is the **direct method**, and it breaks down once $\varepsilon < 1$.  When the policy starts exploiting, action selection becomes biased: the model sends its current-best action to most users and rarely tries alternatives.  Training data accumulates for favored actions and dries up for the rest.  The model becomes well-calibrated where it already exploits, but poorly calibrated elsewhere — and it can never discover it's wrong because it rarely tests its blind spots.

This creates a **feedback loop**: biased data $\to$ biased model $\to$ more biased data.

### Where IPS, SNIPS, and DR each appear

These are all built on importance weighting but serve different purposes:

| Method | When | Purpose |
|--------|------|---------|
| **IPS / SNIPS** | After warm-up (batch) | Evaluate candidate policies offline to select the best $\varepsilon$ |
| **Doubly Robust (DR)** | During each online learning step | Estimate costs for *all* actions (including unchosen) to update the model |

### How Doubly Robust works

On each round, the model needs a cost estimate for **every** action to compute a proper gradient update — not just the one it chose.

A note on notation: VW works in **costs** (lower is better), so click $\to c = 0$ and no click $\to c = 1$.  We use $a'$ to denote a generic action we are computing a cost estimate for (ranging over all $a' \in \{1, \dots, K\}$), and $a_{\text{chosen}}$ for the action that was actually selected on this round.

DR constructs $\hat{c}_{\text{DR}}(a')$ — the corrected cost estimate for action $a'$ — differently depending on whether $a'$ is the action we observed:

**Chosen action** ($a' = a_{\text{chosen}}$):

$$
\hat{c}_{\text{DR}}(a') = \hat{c}_{\text{model}}(a') + \frac{c_{\text{observed}} - \hat{c}_{\text{model}}(a')}{\mu(a \mid x)}
$$

Start with the model's prediction, then add an **IPS-corrected residual**.  If the model was right ($c_{\text{observed}} \approx \hat{c}_{\text{model}}$), the correction is negligible.  If it was wrong, the correction is large — and scaled by $1/\mu$ to account for how likely we were to have selected this action in the first place.

**Unchosen actions** ($a' \neq a_{\text{chosen}}$):

$$
\hat{c}_{\text{DR}}(a') = \hat{c}_{\text{model}}(a')
$$

No observed outcome, so we fall back to the model's current best guess.

With these "filled in" costs for all $K$ actions, the model updates as if it had full-information feedback.

### Why "doubly robust"?

When used as a batch estimator over $N$ logged interactions (same notation as Section 4: each observation $i$ has context $x_i$, chosen action $a_i$, reward $r_i$, and propensity $\mu_i$), the full DR value estimator takes the form:

$$
\hat{V}_{\text{DR}}(\pi) = \frac{1}{N}\sum_{i=1}^{N}\left[\hat{r}(x_i, a_i) + \frac{\pi(a_i \mid x_i)}{\mu(a_i \mid x_i)}\bigl(r_i - \hat{r}(x_i, a_i)\bigr)\right]
$$

For each observation $i$: $\hat{r}(x_i, a_i)$ is the model's reward prediction, and the second term is the IPS-corrected residual — the gap between what actually happened ($r_i$) and what the model expected, scaled by the importance weight.

The name reflects a theoretical guarantee: **the estimate is consistent if *either* the reward model $\hat{r}$ is accurate *or* the propensities $\mu$ are correctly specified**.  Both must be wrong simultaneously for the estimate to fail.

| Approach | What it uses | Weakness |
|----------|-------------|----------|
| **Direct method** | Model prediction $\hat{r}$ only | Selection bias — reinforces existing beliefs |
| **Pure IPS** | Importance weights only, no model | High variance when $\mu$ is small |
| **Doubly Robust** | Model baseline + IPS-corrected residuals | Robust to errors in *either* component |

DR gets the best of both worlds: when the model is accurate, residuals are small and variance is low (like the direct method); when propensities are accurate, the IPS correction fixes model errors (like pure IPS).


## 6. The End-to-End Pipeline

Now we assemble all the pieces into a three-phase pipeline.

---

### Phase 1 — Warm-up (pure exploration, $\varepsilon = 1$)

> *Goal: collect unbiased training data and unbiased logs for OPE.*

1. Initialize a model with $\varepsilon = 1$ (uniform random — every action gets probability $1/K$).
2. For each of $N$ rounds:
   - Observe context $x$
   - Sample action $a$ uniformly; record propensity $\mu(a \mid x) = 1/K$
   - Observe reward $r$; compute cost $c = 1 - r$
   - Update the model using $(x, a, c, \mu)$ via Doubly Robust
   - Save $(x, a, r, \mu)$ to a log for OPE
3. Save the trained model.

> **Why is learning during $\varepsilon = 1$ safe?**  The model *is* updating $\hat{W}$ on every round, which would normally shift the action probabilities.  But with $\varepsilon = 1$, the $\varepsilon$-greedy formula collapses to $\pi(a \mid x) = 1/K$ for all actions — the $\arg\max$ term gets zeroed out.  The model learns quietly in the background, but the exploration policy ignores what it learns.  This gives us two properties: (1) constant, exact propensities ($\mu = 1/K$, which simplifies OPE), and (2) a model with reasonable weight estimates ready for Phase 2.

*Output:* A warm-started model (has learned rough reward estimates from uniform data) **+** a batch of logged interactions with exact propensities.

---

### Phase 2 — Off-Policy Evaluation

> *Goal: pick the best $\varepsilon$ without deploying anything new.*

4. For each candidate $\varepsilon$ (e.g. $0.01, 0.05, 0.1, 0.2, 0.5$):
   - Load the warm-started model with that $\varepsilon$ $\to$ candidate policy $\pi_\varepsilon$
   - For every logged observation, compute $w_i = \pi_\varepsilon(a_i \mid x_i) \,/\, \mu(a_i \mid x_i)$
   - Estimate CTR via IPS and SNIPS
5. Rank candidates by SNIPS score $\to$ select the best $\varepsilon$.

*Output:* The $\varepsilon$ predicted to yield the highest CTR.

---

### Phase 3 — Live deployment (exploit with minimal exploration)

> *Goal: deploy the best policy and keep learning online.*

6. Load the warm-started model with the OPE-selected $\varepsilon$.
7. Run $N$ more rounds of the interaction loop — same four steps as warm-up, but now the policy exploits (e.g. ~99% on its best guess, ~1% exploration).
8. Online learning continues: each round's model update immediately influences the next round's action selection.

*Output:* A policy that outperforms the uniform baseline by matching messages to user segments.

---

> **When does online learning matter?**  During Phase 1 ($\varepsilon = 1$), online learning is a convenience — you could collect logs with a coin flip and train a model on the batch afterward.  Online learning becomes essential in Phase 3 ($\varepsilon < 1$), where $\hat{W}$ directly controls which action the $\arg\max$ selects.  Each update creates a virtuous cycle: better weights $\to$ better action selection $\to$ higher rewards $\to$ better weights.

---

```
 Phase 1: Warm-up          Phase 2: OPE              Phase 3: Live
┌────────────────────┐    ┌────────────────────┐    ┌────────────────────┐
│  ε = 1 (uniform)   │    │  Logged data from  │    │  ε = best (0.01)   │
│                    │    │  Phase 1           │    │                    │
│  Collect data +    │───▶│                    │───▶│  Exploit + learn   │
│  Train model       │    │  IPS / SNIPS eval  │    │  online            │
│  Save logs         │    │  Pick best ε       │    │                    │
└────────────────────┘    └────────────────────┘    └────────────────────┘
     ~9% CTR              (offline, no new            ~12% CTR
   (random baseline)        data needed)           (learned policy)
```


## Scratch Notes

> *Assumes online learning with Doubly Robust cost estimation (`--cb_type dr` in VW).  DR is what allows the model to update weights for all $K$ actions on every round — not just the one that was chosen.  See Section 5 for details.*

Step-by-step walkthrough of what happens, in order:

**Step 1 — Warmup: collect unbiased data.**  We serve actions uniformly at random, so $\mu(a \mid x) = 1/K$ for every action. This is a valid probability distribution: $\sum_{a=1}^{K} \mu(a \mid x) = 1$.  Every action gets equal airtime regardless of context.

**Step 2 — Train the model (during warmup).**  We learn $K$ independent linear models — one per action — each with its own weight vector $\hat{W}[a]$.  For a given context $x$, each model produces a score $\hat{r}(x, a) = \hat{W}[a] \cdot x$.  This is a raw linear output, **not** a probability — it can land outside $[0,1]$ and is not passed through a sigmoid.  The true click probabilities $p(\text{click} \mid x, a) = \sigma(W_{\text{TRUE}}[a] \cdot x)$ are independent Bernoulli means.  The model's output is closely related — with squared loss on cost targets, it converges to a linear approximation of the expected cost $\mathbb{E}[c \mid x, a] = 1 - p(\text{click} \mid x, a)$, so it's **monotonically related** to the true click probability (higher predicted cost ↔ lower click probability).  It doesn't recover the exact probabilities (no sigmoid), but it preserves the ranking — which is all the $\arg\max$ needs.

The loss function is **squared loss (online regression)**, not cross-entropy.  On each round, DR produces a cost target $\hat{c}_{\text{DR}}(a')$ for every action $a'$ (see Section 5), and each action's weights are updated via SGD to minimize the squared error:

$$\hat{W}[a'] \leftarrow \hat{W}[a'] - \eta \cdot \bigl(\hat{W}[a'] \cdot x - \hat{c}_{\text{DR}}(a')\bigr) \cdot x$$

where $\eta$ is the learning rate.  This is why the output is uncalibrated — it's a regression prediction targeting cost values (0 for click, 1 for no click), not a probability estimate.  But since we only use the output for $\arg\max$ ranking, calibration doesn't matter.

**Step 3 — Build a candidate policy.**  Given the trained scores $\hat{r}(x, a)$, the $\varepsilon$-greedy formula converts them into a probability distribution $\pi(\cdot \mid x)$ over the $K$ actions.  The greedy best action (highest $\hat{r}$) gets probability $(1 - \varepsilon + \varepsilon/K)$; every other action gets $\varepsilon/K$.  This sums to 1:

$$\underbrace{(1 - \varepsilon + \varepsilon/K)}_{\text{best action}} + \underbrace{(K-1) \cdot \varepsilon/K}_{\text{all others}} = 1 - \varepsilon + \varepsilon/K + \varepsilon - \varepsilon/K = 1$$

For $\varepsilon = 0.05$, $K = 4$: best action gets $0.9625$, each other gets $0.0125$, total $= 1$.

**Step 4 — Compute importance weights.**  For each logged observation $i$, we compare how much $\pi$ favors the action that was taken vs. how much $\mu$ did: $w_i = \pi(a_i \mid x_i) / \mu(a_i \mid x_i)$.  Computing $\pi(a_i \mid x_i)$ is deterministic — just evaluate the $\varepsilon$-greedy formula for the logged context and action.  No sampling needed.

**Step 5 — SNIPS picks the best $\varepsilon$.**  For each candidate $\varepsilon$, repeat steps 3–4 and compute $\hat{V}_{\text{SNIPS}}$.  The reweighting effectively reconstructs what $\pi$'s data would have looked like: observations of actions that $\pi$ favors get amplified, and observations of actions $\pi$ avoids get suppressed.  Since the environment's rewards don't depend on which policy chose the action, a dataset with $\pi$'s action frequencies gives $\pi$'s expected reward.  Pick the $\varepsilon$ with the highest SNIPS score.

**Step 6 — Deploy and keep learning.**  Load the model with the best $\varepsilon$ and run live.  Now the model's scores directly control which action gets served, and each new observation updates $\hat{W}$ — creating a feedback loop that improves over time.

# SCRATCH NOTES